# 03 — Text Analysis & LLM-Based Annotation

This notebook uses the Anthropic API (Claude) to score each post on
linguistic and behavioral dimensions that cannot be captured by simple
statistics. Claude reads each Hebrew post and returns structured scores
that are then merged with the clean dataset for predictive modeling.

**Input:** `data/clean_posts.csv`
**Output:** `data/clean_posts_with_text.csv`

**Sections:**
1. Configuration & Setup
2. Load Data
3. Prompt Design & Single Post Test
4. Score All Posts (with incremental saving)
5. Manual Validation
6. Parse & Merge Results
7. Export

## 1. Configuration & Setup

In [40]:
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import anthropic

# Load API key from .env file
load_dotenv()

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE      = Path("../data/clean_posts.csv")
OUTPUT_FILE     = Path("../data/clean_posts_with_text.csv")
CHECKPOINT_FILE = Path("../data/text_scores_checkpoint.json")

# ── Model ──────────────────────────────────────────────────────────────────
MODEL      = "claude-haiku-4-5"
MAX_TOKENS = 600

# ── API client ─────────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("✓ API client ready")
print(f"  Model:  {MODEL}")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")

✓ API client ready
  Model:  claude-haiku-4-5
  Input:  ../data/clean_posts.csv
  Output: ../data/clean_posts_with_text.csv


## 2. Load Data

In [41]:
df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df)} posts")
print(f"Columns: {df.columns.tolist()}")
print()
df[["post_id", "title", "date", "engagement_rate", "view_through_rate"]].head(3)

Loaded 97 posts
Columns: ['post_id', 'page_id', 'page_name', 'title', 'duration_sec_', 'publish_time', 'permalink', 'post_type', 'data_comment', 'date', 'comments', 'distribution', 'impressions', 'interactions', 'reactions', 'saves', 'shares', 'views', 'viewers', 'net_follows', 'average_seconds_viewed', 'seconds_viewed', 'time', 'hour', 'weekday', 'post_origin', 'text_length_chars', 'text_length_words', 'has_hashtag', 'hashtag_count', 'mentions_children', 'engagement_rate', 'repeat_view_rate', 'view_through_rate', 'post_category']



,post_id,title,date,engagement_rate,view_through_rate
0,10234810932441995,חופשה עם ילדים זאת מעין חצי חופשה. \n\nלא מקבל...,2025-08-14,0.056085,0.834921
1,10238066919799644,מלחמה וסטרס עושות משהו מעניין לזמן.\n\nהן מותח...,2026-03-19,0.105033,0.947484
2,10237996040547707,מה אנחנו באמת רוצים כשאנחנו חושקים במישהו?\n\n...,2026-03-16,0.011610,0.918539


## 3. Prompt Design & Single Post Test

We first design the scoring prompt and test it on a single post
before running it on the full dataset. This lets us verify the
output format and scoring quality manually.

The prompt instructs Claude to:
- Read a Hebrew Facebook post
- Score it on 12 behavioral and linguistic dimensions
- Return scores as a valid JSON object only — no extra text

In [42]:
SYSTEM_PROMPT = """You are an expert in behavioral psychology and linguistic analysis.
You are analyzing Hebrew Facebook posts written by a single author — a behavioral
neuroscientist and published novelist — who uses social media to share personal
reflections and promote her book.

Your task is to score each post on the dimensions below.
Return ONLY a valid JSON object with no additional text, preamble, or markdown formatting.

Scoring dimensions:

1. emotional_valence: Overall emotional tone of the post
   1=very negative, 2=negative, 3=neutral, 4=positive, 5=very positive

2. emotional_intensity: How strong and pervasive the emotion is throughout the post
   1=very mild or flat, 2=mild, 3=moderate, 4=strong, 5=very strong and intense

3. tone: Primary tone of the post. Choose one:
   - reflective: personal experience, inner world, first-person emotional writing
   - recommendation: suggesting something to the audience (a book, series, idea, place)
   - book_accomplishment: announcing or celebrating something about the author's own book
   - occasion_anchored: the post is structured around a specific date or external event

4. marketing_mention: Does the post explicitly mention the book is available for purchase,
   name bookstores (סטימצקי, צומת ספרים, עברית), or include a direct call to buy?
   0=no, 1=yes

5. occasion_mention: Does the post reference a specific occasion such as
   ראש השנה, יום האישה, יום הולדת, חגים, מלחמה, or another named event?
   0=no, 1=yes

6. occasion_relevance: If occasion_mention=1, how central is the occasion to the post?
   1=passing mention only, 2=moderate role, 3=the post is primarily about the occasion
   If occasion_mention=0, return 0

7. dominant_pronoun: Which pronoun dominates the overall post body?
   - I: post is primarily first-person (אני, שלי, לי, אותי)
   - You: post primarily addresses the reader (את, אתה, אתם, שלך)
   - We: post uses collective voice (אנחנו, שלנו)
   - Mixed: no single pronoun dominates

8. opening_pronoun: What pronoun appears in the very first sentence?
   Options: I / You / We / None
   None = the first sentence does not clearly use any of these pronouns

9. question_presence: Does the post contain a direct question to the audience?
   0=no, 1=yes

10. list_structure: Is the post structured as a numbered or bullet list,
    or does it use clearly repeated parallel sentence structures?
    0=no, 1=yes

11. narrative_arc: Does the post tell a story with movement or change?
    1=no arc, observation or reflection only
    2=partial arc, some movement but no clear resolution
    3=clear arc with beginning, middle, and end or resolution

12. personal_vulnerability: Does the author disclose something that feels
    risky, private, or emotionally exposed?
    1=not vulnerable, surface level
    2=moderately vulnerable, personal but not deeply exposed
    3=highly vulnerable, the author shares something intimate or difficult

13. parenthood_theme: Is parenthood, motherhood, or the author's
    children a meaningful theme in this post?
    0 = not mentioned, or only a passing reference
    1 = children or parenthood play a meaningful role in the post

14. shared_context: Does the post tap into an experience that a significant
    portion of the Israeli audience is likely going through at the same time
    as the post was published? Use the post date provided to judge relevance.
    Examples: summer vacation with kids in August, war anxiety during conflict,
    Jewish holidays, back to school, national events.
    0 = no, the content is timeless or purely personal
    1 = yes, it reflects a shared current social or cultural experience
    Key criterion: temporal co-occurrence — the audience is experiencing
    this specific thing at this specific moment, not just that the theme
    is universally relatable. A post about a grandmother dying scores 0
    (death is universal but not temporally shared); a post about war
    anxiety during an ongoing conflict scores 1.

15. tag_count: How many people or pages are tagged in this post?
    Tagged users appear as proper names (not preceded by @) and are typically
    found at the end of the post but can appear anywhere. They are usually
    names of real people or pages being mentioned or credited.
    Return an integer: 0, 1, 2, 3, etc.

16. explanation: A brief 2-3 sentence explanation of your scores,
    focusing on the dimensions that were most ambiguous or interesting.
    Write in English.

Return your response as a single JSON object with exactly these keys:
emotional_valence, emotional_intensity, tone, marketing_mention,
occasion_mention, occasion_relevance, dominant_pronoun, opening_pronoun,
question_presence, list_structure, narrative_arc, personal_vulnerability,
parenthood_theme, shared_context, tag_count, explanation
"""

In [43]:
def score_post(post_text, post_date=None):
    """Send a single post to Claude and return the parsed JSON scores."""
    date_line = f"Post date: {post_date}\n\n" if post_date else ""
    message = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": f"Please score the following Hebrew Facebook post:\n\n{date_line}Post text:\n{post_text}"
            }
        ]
    )

    raw = message.content[0].text.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1]
        raw = raw.rsplit("```", 1)[0].strip()

    return json.loads(raw)

In [44]:
# ── Single post test ────────────────────────────────────────────────────────
# Test on the first post in the dataset
test_post = df.iloc[0]

print("POST TEXT:")
print("─" * 60)
print(test_post["title"])
print("─" * 60)
print(f"Date: {test_post['date']}  |  Type: {test_post['post_type']}")
print(f"Engagement rate: {test_post['engagement_rate']:.3f}  |  VTR: {test_post['view_through_rate']:.3f}")
print()

# Score it
result = score_post(test_post["title"], post_date=test_post["date"])

print("CLAUDE'S SCORES:")
print("─" * 60)
for key, value in result.items():
    if key != "explanation":
        print(f"  {key:<25} {value}")
print()
print("EXPLANATION:")
print(result.get("explanation", "No explanation provided"))

POST TEXT:
────────────────────────────────────────────────────────────
חופשה עם ילדים זאת מעין חצי חופשה. 

לא מקבלים באמת את כל החופש שהיית רוצה ורוב הזמן את די עובדת, אבל כן יש רגעים קסומים. 

כשדניאל קופץ וצוחק בבריכה. כשאני רואה איך לאט לאט נבנה לו הביטחון והוא לא מפחד יותר להיכנס ואפילו רוצה לקפוץ. כשאנחנו מלמדים אותו מזה קייט ופתאום הוא מחפש קייטים בים. 

כשאני מביטה בקפלים המתוקים שברגליים השמנמנות שלו ומנסה לשתות כל רגע כזה לפני שהוא פשוט חולף. 

כי למרות שכל ארוחה במסעדה היא סימפוניה של ״אני רוצה גם״ וכל נסיעה לים יכולה להסתיים בטנטרום כשרוצים כבר ללכת, אני כל הזמן מזכירה לעצמי שזה עוד רגע עובר והם כבר יהיו גדולים ואולי כבר לא יצטרכו אותי כל-כך. 

כי הזמן יש בו משהו מתעתע. הוא נראה כאילו לעולם לא יגמר כשהוא בהתקף טנטרום אבל אז בן רגע הוא פתאם נהיה גדול וכבר לא התינוק הקטן והשמנמן שהחזקתי בידיים לפני רגע. 

אז זאת מעין חצי חופשה כזאת שדורשת יותר עבודה סביב הילד אבל גם ממלאת אותי בו.

בתמונה - כלי הנשק הסודי שלי, גלידה.
──────────────────────────────────────────────────────────

**Manual check:** Read the post above and verify the scores make sense.
If something looks wrong, adjust the prompt in the cell above and re-run the test
before proceeding to score all posts.

## 4. Score All Posts (with incremental saving)

We score all posts one by one, saving results to a checkpoint file after
each post. If the process is interrupted, re-running this cell will skip
already-scored posts and continue from where it left off.

In [45]:
# Load existing checkpoint if available
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    print(f"Loaded checkpoint: {len(checkpoint)} posts already scored")
else:
    checkpoint = {}
    print("No checkpoint found — starting fresh")

# Score all posts
errors = []

for idx, row in df.iterrows():
    post_id = str(row["post_id"])

    # Skip if already scored
    if post_id in checkpoint:
        continue

    # Skip posts with no text
    if pd.isna(row["title"]) or str(row["title"]).strip() == "":
        errors.append({"post_id": post_id, "error": "empty title"})
        continue

    try:
        scores = score_post(row["title"], post_date=row["date"])
        checkpoint[post_id] = scores

        # Save checkpoint after every post
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)

        print(f"✓ [{idx+1}/{len(df)}] post_id {post_id} — tone: {scores.get('tone')} | valence: {scores.get('emotional_valence')}")

        # Small delay to avoid rate limiting
        time.sleep(0.5)

    except Exception as e:
        print(f"✗ [{idx+1}/{len(df)}] post_id {post_id} — ERROR: {e}")
        errors.append({"post_id": post_id, "error": str(e)})

print()
print(f"Done. {len(checkpoint)} posts scored, {len(errors)} errors.")
if errors:
    print("Errors:")
    for e in errors:
        print(f"  post_id {e['post_id']}: {e['error']}")

Loaded checkpoint: 97 posts already scored

Done. 97 posts scored, 0 errors.


### Detect Missing Dimensions

When new scoring dimensions are added after some posts have already been scored,
existing checkpoint entries won't have those dimensions. This cell identifies
which posts are missing any of the expected dimensions and marks them for re-scoring.

Re-running the scoring loop below will then re-score only those posts.

In [46]:
# All dimensions we expect in every scored post
EXPECTED_DIMS = [
    "emotional_valence", "emotional_intensity", "tone",
    "marketing_mention", "occasion_mention", "occasion_relevance",
    "dominant_pronoun", "opening_pronoun", "question_presence",
    "list_structure", "narrative_arc", "personal_vulnerability",
    "parenthood_theme", "shared_context", "tag_count", "explanation"
]

# Find posts missing any expected dimension
incomplete = [
    post_id for post_id, scores in checkpoint.items()
    if any(dim not in scores for dim in EXPECTED_DIMS)
]

if incomplete:
    print(f"Found {len(incomplete)} post(s) missing one or more dimensions:")
    for post_id in incomplete:
        missing = [d for d in EXPECTED_DIMS if d not in checkpoint[post_id]]
        print(f"  post_id {post_id} — missing: {missing}")
    print()
    print("Removing incomplete entries from checkpoint so they get re-scored...")
    for post_id in incomplete:
        del checkpoint[post_id]
    print(f"Checkpoint now has {len(checkpoint)} complete entries.")
    print("Re-run the scoring loop below to fill in the missing posts.")
else:
    print(f"All {len(checkpoint)} checkpoint entries have the expected dimensions. Nothing to re-score.")


All 97 checkpoint entries have the expected dimensions. Nothing to re-score.


## 5. Manual Validation

Before merging scores into the dataset, manually inspect a sample of posts
to verify that Claude's scores match your intuition as the author.

This is an important methodological step — it establishes that the annotation
is valid before using it in any predictive model.

In [47]:
# Display a random sample of 5 posts with their scores for manual validation
sample = df.sample(5, random_state=42)

for _, row in sample.iterrows():
    post_id = str(row["post_id"])
    scores  = checkpoint.get(post_id, {})

    print("=" * 65)
    print(f"POST ID: {post_id}  |  Date: {row['date']}  |  Type: {row['post_type']}")
    print(f"Engagement rate: {row['engagement_rate']:.3f}  |  VTR: {row.get('view_through_rate', 'N/A')}  |  Category: {row['post_category']}")
    print()

    # Structural features
    print("STRUCTURAL FEATURES:")
    print(f"  has_hashtag:       {row.get('has_hashtag', 'N/A')}  |  hashtag_count: {row.get('hashtag_count', 'N/A')}")
    print(f"  mentions_children: {row.get('mentions_children', 'N/A')}")
    print()

    # Full post text
    print("FULL TEXT:")
    print(str(row["title"]))
    print()

    # LLM scores
    print("LLM SCORES:")
    for key, value in scores.items():
        if key != "explanation":
            print(f"  {key:<25} {value}")
    print()
    print("EXPLANATION:")
    print(scores.get("explanation", "N/A"))
    print()


POST ID: 10235889728531223  |  Date: 2025-10-21  |  Type: Content
Engagement rate: 0.054  |  VTR: 0.8786610878661087  |  Category: Low Performance

STRUCTURAL FEATURES:
  has_hashtag:       1  |  hashtag_count: 1
  mentions_children: 0

FULL TEXT:
תודה ל חגית שושני על הסקירה המעמיקה:) 

#בין_המסדרונות הוא ספר על הציפיות והמגבלות שהחברה שמה עלינו והרצון שלנו לפרוץ אותן ולגלות מי אנחנו בעצם. 

כדי להתקרב לעצמה, אלי גיבורת הספר, נאלצת סופסוף להתעמת עם האמת.

LLM SCORES:
  emotional_valence         4
  emotional_intensity       2
  tone                      book_accomplishment
  marketing_mention         0
  occasion_mention          0
  occasion_relevance        0
  dominant_pronoun          Mixed
  opening_pronoun           None
  question_presence         0
  list_structure            0
  narrative_arc             1
  personal_vulnerability    1
  parenthood_theme          0
  shared_context            0
  tag_count                 1

EXPLANATION:
This post announces a favorable review 

In [ ]:
# Look up a specific post by index (change the number to inspect any post)
INSPECT_INDEX = 16  # ← change this to inspect a different post

row     = df.iloc[INSPECT_INDEX]
post_id = str(row["post_id"])
scores  = checkpoint.get(post_id, {})

print("=" * 65)
print(f"POST ID: {post_id}  |  Date: {row['date']}  |  Type: {row['post_type']}")
print(f"Engagement rate: {row['engagement_rate']:.3f}  |  VTR: {row.get('view_through_rate', 'N/A')}  |  Category: {row['post_category']}")
print()

# Structural features
print("STRUCTURAL FEATURES:")
print(f"  has_hashtag:       {row.get('has_hashtag', 'N/A')}  |  hashtag_count: {row.get('hashtag_count', 'N/A')}")
print(f"  mentions_children: {row.get('mentions_children', 'N/A')}")
print()

# Full post text
print("FULL TEXT:")
print(row["title"])
print()

# LLM scores
print("LLM SCORES:")
for key, value in scores.items():
    if key != "explanation":
        print(f"  {key:<25} {value}")
print()
print("EXPLANATION:")
print(scores.get("explanation", "N/A"))


POST ID: 10237493493784352  |  Date: 2026-02-15  |  Type: Reel
Engagement rate: 0.049  |  VTR: 0.7307692307692307  |  Category: Low Performance

STRUCTURAL FEATURES:
  has_hashtag:       1  |  hashtag_count: 5
  has_tag:           N/A
  mentions_children: 0

FULL TEXT:
״קודם כל אני רוסייה. 
לא אמרתי לעצמי שאני טובה בכתיבה.״

דיברנו על להיות רוסיות, 
על הרדיפה העצמית, 
על כמה העבר שלנו מכתיב מי אנחנו בהווה,
ועל מה שגרם לי לכתוב את ״בין המסדרונות״.

מוזמנים להאזין או לצפות בראיון המלא בפודקאסט ״בתוך הראש הסגול שלי״ עם אירה שוסטר המקסימה:)

#זהות #דוראחדוחצי #בין_המסדרונות #רוסיותבליחושהומורוחבריהן #כתיבה

LLM SCORES:
  emotional_valence         4
  emotional_intensity       3
  tone                      book_accomplishment
  marketing_mention         0
  occasion_mention          0
  occasion_relevance        0
  dominant_pronoun          I
  opening_pronoun           I
  question_presence         0
  list_structure            1
  narrative_arc             2
  personal_vulnerability    2

## 6. Parse & Merge Results

Convert the checkpoint dictionary into a dataframe and merge with the
original clean dataset. Each score becomes a new column.

In [ ]:
# Convert checkpoint to dataframe
scores_df = pd.DataFrame.from_dict(checkpoint, orient="index")
scores_df.index.name = "post_id"
scores_df = scores_df.reset_index()
scores_df["post_id"] = scores_df["post_id"].astype(df["post_id"].dtype)

print(f"Scores dataframe: {len(scores_df)} rows x {len(scores_df.columns)} columns")
print(f"Score columns: {[c for c in scores_df.columns if c != 'post_id']}")
scores_df.head(3)

In [ ]:
# Merge with clean dataset
df_enriched = df.merge(scores_df, on="post_id", how="left")

print(f"Enriched dataset: {len(df_enriched)} rows x {len(df_enriched.columns)} columns")
print(f"Posts with scores: {df_enriched['emotional_valence'].notna().sum()}")
print(f"Posts missing scores: {df_enriched['emotional_valence'].isna().sum()}")

In [ ]:
# Quick summary of score distributions
score_cols = [
    "emotional_valence", "emotional_intensity", "tone",
    "marketing_mention", "occasion_mention", "occasion_relevance",
    "dominant_pronoun", "opening_pronoun", "question_presence",
    "list_structure", "narrative_arc", "personal_vulnerability",
    "parenthood_theme",
    "shared_context",
    "tag_count"
]

print("Score distributions:")
print()
for col in score_cols:
    if col in df_enriched.columns:
        print(f"{col}:")
        print(df_enriched[col].value_counts().sort_index().to_string())
        print()

## 7. Export

Save the enriched dataset. This file is the input for `04_modeling.ipynb`.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_enriched.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"✓ Exported {len(df_enriched)} posts to {OUTPUT_FILE}")
print(f"  Total columns: {len(df_enriched.columns)}")
print(f"  New text feature columns: {len(score_cols)}")